# Удаление адаптеров и фильтрация целых ридов по качеству и длине — ERP003950 mouse IGH, ветка fastp

Схема обработки:

1. `cutadapt` удаляет только явно заданные адаптеры Illumina, без обрезки по качеству и фильтра по длине.
2. `fastp` фильтрует риды целиком: Q30 (`-q 30`, основания ниже Q30 считаются некачественными), `-u 40` и минимальная длина пары 250.
3. Удаление адаптеров и poly-G в `fastp` отключено; параметры обрезки качества `--cut_*` не используются.
4. Праймеры FR1/CH1 обрабатываются отдельно.

Существующие результаты без соответствующего JSON от `fastp` считаются несовместимыми и не переиспользуются и не удаляются автоматически.


In [ ]:
import os, sys, sysconfig, shutil, subprocess, time, gzip, json
from pathlib import Path
_ENV_CANDIDATES=['/opt/conda/envs/bcr_env','/Users/epishkin/mamba/envs/bcr_env']
_CONDA_ENV=next((x for x in _ENV_CANDIDATES if os.path.isdir(x+'/bin')), _ENV_CANDIDATES[-1])
os.environ['PATH'] = _CONDA_ENV + '/bin:' + os.environ.get('PATH', '')
os.environ['PYTHONNOUSERSITE'] = '1'
sys.path[:] = [p for p in sys.path if '/data/user/epishkin/.local' not in p]
for _site in [_CONDA_ENV + '/lib/python3.11/site-packages', sysconfig.get_path('purelib')]:
    if os.path.isdir(_site) and _site not in sys.path: sys.path.insert(0, _site)
if Path('/data/user/epishkin').is_dir():
    os.environ['HOME'] = '/data/user/epishkin'
    os.environ['XDG_CONFIG_HOME'] = '/data/user/epishkin/.config'
    os.makedirs(os.environ['XDG_CONFIG_HOME'], exist_ok=True)


In [ ]:
START=Path.cwd().resolve()
LOCAL_REPO=next((p for p in (START,*START.parents) if (p/'raw'/'ERP003950').is_dir()),None)
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'raw'/'ERP003950').is_dir() and LOCAL_REPO is not None: VOLUME=LOCAL_REPO
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10
QUALITY_PHRED = 30
UNQUALIFIED_PERCENT_LIMIT = 40
MIN_LENGTH = 250
ILLUMINA_ADAPTER_R1 = 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ILLUMINA_ADAPTER_R2 = 'GATCGGAAGAGCACACGTCTGAACTCCAGTCAC'

def tool(name):
    p=shutil.which(name)
    if not p: raise FileNotFoundError(name)
    return p

def fqcount(path):
    with gzip.open(path,'rt') as h: n=sum(1 for _ in h)
    if n%4: raise ValueError(f'truncated FASTQ: {path}')
    return n//4

def run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    started=time.monotonic(); print('[run]', ' '.join(map(str,cmd)), flush=True)
    with open(stdout_log,'w') as out, open(stderr_log,'w') as err:
        proc=subprocess.Popen([str(x) for x in cmd],stdout=out,stderr=err,text=True)
        print(f'PID={proc.pid}',flush=True)
        while proc.poll() is None:
            sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
            print(f'PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True)
            time.sleep(heartbeat)
    if proc.returncode: raise RuntimeError(f'rc={proc.returncode}; see {stderr_log}')

def run_adapter_filter(volume, dataset, force=False):
    vol=Path(volume); src=vol/'raw'/dataset; base=vol/'results'/'ERP003950'/'trimmed'
    out=base/'fastq'; logs=base/'logs'; reports=base/'fastp_reports'; tmp=base/'adapter_only_tmp'
    if not src.is_dir(): raise FileNotFoundError(src)
    if force and base.exists(): shutil.rmtree(base)
    for d in (out,logs,reports,tmp): d.mkdir(parents=True,exist_ok=True)
    pairs=sorted({f.name.rsplit('_',1)[0] for f in src.glob('*_1.fastq.gz')})
    print(f'[adapter_filter] {dataset}: {len(pairs)} pairs; Q={QUALITY_PHRED}; unqualified<={UNQUALIFIED_PERCENT_LIMIT}%; minlen={MIN_LENGTH}')
    for bn in pairs:
        r1=src/f'{bn}_1.fastq.gz'; r2=src/f'{bn}_2.fastq.gz'
        o1=out/f'{bn}_1.trim.fastq.gz'; o2=out/f'{bn}_2.trim.fastq.gz'; jf=reports/f'{bn}.fastp.json'; hf=reports/f'{bn}.fastp.html'
        if o1.exists() and o2.exists() and jf.exists(): print(f'[{bn}] complete, skip'); continue
        if o1.exists() or o2.exists() or jf.exists():
            raise RuntimeError(f'{bn}: incompatible/partial existing outputs; preserve them or rerun explicitly with force=True')
        a1=tmp/f'{bn}_1.adapter.fastq.gz'; a2=tmp/f'{bn}_2.adapter.fastq.gz'
        run_visible([tool('cutadapt'),'--times',str(ADAPTER_TIMES),'-O',str(ADAPTER_MIN_OVERLAP),'--compression-level','1','-a',ILLUMINA_ADAPTER_R1,'-A',ILLUMINA_ADAPTER_R2,'--json',logs/f'{bn}.cutadapt.json','-o',a1,'-p',a2,r1,r2],logs/f'{bn}.cutadapt.stdout.log',logs/f'{bn}.cutadapt.stderr.log',[a1,a2])
        run_visible([tool('fastp'),'-i',a1,'-I',a2,'-o',o1,'-O',o2,'-q',str(QUALITY_PHRED),'-u',str(UNQUALIFIED_PERCENT_LIMIT),'-l',str(MIN_LENGTH),'--disable_adapter_trimming','--disable_trim_poly_g','-w','4','-j',jf,'-h',hf],logs/f'{bn}.fastp.stdout.log',logs/f'{bn}.fastp.stderr.log',[o1,o2])
        a1.unlink(); a2.unlink()
        if fqcount(o1)!=fqcount(o2): raise RuntimeError(f'{bn}: output mates differ')
    summary={'quality_semantics':'fastp whole-read filter; no quality trimming','qualified_quality_phred':QUALITY_PHRED,'unqualified_percent_limit':UNQUALIFIED_PERCENT_LIMIT,'minimum_length':MIN_LENGTH,'pairs':{}}
    for jf in sorted(reports.glob('*.fastp.json')):
        d=json.loads(jf.read_text()); s=d['summary']; summary['pairs'][jf.name.removesuffix('.fastp.json')]={'before_reads':s['before_filtering']['total_reads']//2,'after_reads':s['after_filtering']['total_reads']//2,'low_quality_reads':d['filtering_result']['low_quality_reads'],'too_short_reads':d['filtering_result']['too_short_reads']}
    (base/'filter_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
    print('[adapter_filter] DONE',len(summary['pairs']))


## Запуск

`force=False` не перезаписывает существующие результаты. Для замены несовместимых результатов сначала сохраните их, затем явно установите `force=True`.


In [ ]:
run_adapter_filter(VOLUME, 'ERP003950', force=False)
